# Quantum autoencoder benchmark

Six qubits, two encodings, seven classical controls and two LHCO simulated signal topologies. Training and selection use background only.

Start in **inspect** mode to read saved results without training. **reproduce** runs seed 0 with the frozen selection; **study** runs fresh selection and 15 seeds. Training uses the resumable `tools/colab_run.py` workflow.

For training select a GPU runtime. Allow hours for a full study; free-tier resources and runtime are not guaranteed. The two feature files total about 80 MB, plus about 80 MB for saved scores. No raw-particle file is required.

## 1. Checkout and environment

Colab uses `/content/repo` if present, otherwise clones the public repository. To test unpublished changes, upload and extract the reviewed checkout there first. Existing checkouts are never automatically pulled. Local users should install `requirements.txt` in a virtual environment first.

Colab retains its supplied PyTorch/CUDA. Quantum-library versions are pinned; other dependencies may differ from the historical session. This is a compatibility setup, not an exact recreation. If installation requests a restart, restart and rerun from the beginning.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util
IN_COLAB = bool(importlib.util.find_spec('google')) and importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    ROOT = Path('/content/repo')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/HaronJadid/quantum-autoencoder-hep-anomaly-detection.git', str(ROOT)], check=True)
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/run_study.py').exists()), None)
if ROOT is None or not (ROOT / 'tools/colab_run.py').exists():
    raise RuntimeError('Open this notebook inside the checkout, or extract it to /content/repo.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
if (ROOT / '.git').exists():
    subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
    subprocess.run(['git', 'status', '--short'], check=True)
else:
    print('Archive checkout: no Git metadata. The training manifest records source hashes.')
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pennylane==0.45.1', 'qiskit==2.5.2', 'pylatexenc', 'tables', 'pandas', 'matplotlib', 'scikit-learn', 'scipy'], check=True)
import torch, importlib.metadata as metadata
print(sys.version)
for package in ('torch', 'numpy', 'pennylane', 'qiskit', 'scikit-learn', 'scipy', 'pandas', 'tables', 'matplotlib', 'pylatexenc'):
    print(package, metadata.version(package))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')


## 2. Choose a mode

- `inspect`: saved 15-seed study; no training or data download.
- `reproduce`: seed 0, frozen selection, both signals, no separate ablation. One seed cannot estimate variability.
- `study`: final-v2 selection and seeds 0–14 on a fixed split. Quantum selection uses 100 epochs on 25k training events; classical learning-rate selection uses the full sample and epoch budget over three initialisations. The historical ablation is skipped.

Both training modes use final-v2 event-weighted validation and a fixed partition. New output folders protect the reported results. Drive mounting preserves completed chunks across Colab resets. Resume with identical code/settings/environment; otherwise use a new RUN_NAME. An unfinished chunk restarts from its beginning.

In [ ]:
MODE = 'inspect'  # inspect, reproduce, study
RUN_NAME = 'qae-final-v2-release'
USE_DRIVE = True
if MODE not in {'inspect', 'reproduce', 'study'}:
    raise ValueError('Unknown MODE')
if MODE != 'inspect' and not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training.')
if MODE != 'inspect' and IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_BASE = Path('/content/drive/MyDrive/QAE-runs')
else:
    RUN_BASE = ROOT / 'runs'
OUT = (RUN_BASE / RUN_NAME / MODE).resolve()
RESULTS = ROOT / 'results/final-v2/metrics.json' if MODE == 'inspect' else OUT / 'metrics.json'
print('Mode:', MODE, 'Results:', RESULTS)


## 3. Run

Inspect mode skips training. The runner verifies dataset checksums and circuit equivalence, saves selection and chunks, and generates figures. Failures stop this cell. If Zenodo is unreachable, resolve connectivity and retry; an error alone does not establish a global outage.

In [ ]:
if MODE != 'inspect':
    command = [sys.executable, '-u', 'tools/colab_run.py', '--device', 'cuda', '--seeds', '1' if MODE == 'reproduce' else '15', '--chunk', '1' if MODE == 'reproduce' else '5', '--save-scores', '--out', str(OUT)]
    if MODE == 'reproduce':
        command += ['--selection-from', str(ROOT / 'results/final-v2/selection.json'), '--protocol', 'final-v2', '--split-seed', '0', '--selection-seed', '0', '--select-epochs', '100', '--no-ablation']
    else:
        command += ['--protocol', 'final-v2', '--split-seed', '0', '--selection-seed', '0', '--select-epochs', '100', '--no-ablation']
    from datetime import datetime, timezone
    OUT.mkdir(parents=True, exist_ok=True)
    log_path = OUT / ('launch-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.log')
    print('Log:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        log.write(repr(command) + '\n')
        log.flush()
        with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True,
                              encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        log.write(f'\nExit status: {returncode}\n')
    if returncode:
        raise RuntimeError(f'Runner exited with status {returncode}. See the output above and {log_path}. Completed results are preserved.')
else:
    print('Using the saved study; no training requested.')


## 4. Results and comparison

The report generator produces the detailed release report; a separate generator produces the compact README summary. New runs are displayed/exported separately, never automatically promoted to reported results. The compression ceiling concerns a sampled density matrix, not an anomaly-detection guarantee.

In [ ]:
import json
import importlib, src.report
importlib.reload(src.report)
render = src.report.render
from IPython.display import Markdown, display, Image
res = json.loads(RESULTS.read_text())
report_text = render(res)
display(Markdown(report_text))
if MODE != 'inspect':
    (OUT / 'report.md').write_text(report_text, encoding='utf-8')
if MODE == 'reproduce':
    reference = json.loads((ROOT / 'results/final-v2/metrics.json').read_text())
    rows = ['| Model | Saved seed-0 AUC | New AUC | Difference |', '|---|---:|---:|---:|']
    for name, runs in res['per_seed'].items():
        old = reference['per_seed'][name][0]['auc']
        new = runs[0]['auc']
        rows.append(f'| {name} | {old:.8f} | {new:.8f} | {new-old:+.8f} |')
    display(Markdown('\n'.join(rows)))
    print('Matching AUCs checks these metrics, not model weights or every data array.')


In [ ]:
for name in ('auc.png', 'roc.png', 'generalisation.png', 'scores.png', 'sculpting.png', 'training_curves.png', 'circuit.png'):
    path = RESULTS.parent / 'figures' / name
    if not path.exists():
        raise FileNotFoundError(f'Missing figure: {path}')
    print(name)
    display(Image(filename=str(path)))


## 5. Export

A training archive contains metrics, selection, resume manifest, figures, report and saved scores. Keep it for review before replacing reported results. Inspect mode creates no archive.

In [ ]:
if MODE != 'inspect':
    import shutil
    archive = shutil.make_archive(str(OUT) + '-bundle', 'zip', root_dir=OUT)
    print('Saved:', archive)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
